# Notebook 2: The tool-using Agent

### Purpose

A model knows a lot of things, but it cannot perform tasks on its own. It cannot inspect a set of papers or search for current trends unless the application supplies that capability. Tools close that gap. A tool is how an agent reaches outside its own head.

By the end you will be able to **explain how an agent extends it's capabilities via tool use**, becuase you'lle connect multiple tools and etach the model make decisions which tool to call.

## Outcomes

- Web search tool 
- Document retrieval tool
- Structured output tool

# Prerequisites

- Finished Notebook 1
- Same setup: runs with `mock` mode with no key; switch `PROVIDER` when ready 

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ms-cc-org/AGENTIC-AI-Workshop/blob/main/notebooks/02_tool_using_agent.ipynb)

# STEP 1 - Setup (Same block as Notebook 1)

In [ ]:
#  SETUP cell. This is for a provider-agnostic LLM client.
#  Read this cell; every notebook uses this same adapter.
#  Switch providers by changing PROVIDER.
#  Nothingelse in the notebook changes. an agent is a pattern, not a vendor.

# In Google Colab this cell installs the packages. Locally, run once.
try:
    import google.colab
    !pip -q install anthropic openai
except Exception:
    pass

import os, json

PROVIDER = "openai"   # "anthropic" | "openai" | "mock"
                      # "mock" runs this notebook with NO API key, 

MODELS = {
    "anthropic": "claude-haiku-4-5-20251001",  # cheapest current Claude model
    "openai":    "gpt-5.4-nano", # cheapest current GPT model
}

# API keys
# In Colab: click the key icon (left sidebar) -> add ANTHROPIC_API_KEY or
# OPENAI_API_KEY, then uncomment the line below and your specific provider to load them.
#from google.colab import userdata
#os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
#os.environ["OPENAI_API_KEY"] = userdata.get("MY_API_KEY")

# Normalized shapes we use everywhere (so the agent code never mentions a vendor):
#   message: {"role":"user","content":str}
#            {"role":"assistant","content":str|None,"tool_calls":[{id,name,args}]}
#            {"role":"tool","tool_call_id":id,"name":name,"content":str}
#   tool:    {"name":str,"description":str,"parameters":<json-schema>}
#   reply:   {"text":str,"tool_calls":[{id,name,args}],"stop_reason":str}


_MOCK_SCRIPT = []
def mock_reset(script):
    global _MOCK_SCRIPT; _MOCK_SCRIPT = list(script)
def _mock_call(messages, tools):
    if _MOCK_SCRIPT:
        kind, payload = _MOCK_SCRIPT.pop(0)
        if kind == "tool":
            return {"text":"", "tool_calls":[{"id":"mock_"+payload["name"],
                    "name":payload["name"], "args":payload["args"]}], "stop_reason":"tool_use"}
        return {"text":payload, "tool_calls":[], "stop_reason":"end"}
    last = next((m for m in reversed(messages) if m["role"] in ("user","tool")), {"content":""})
    return {"text": f"[mock reply to] {str(last.get('content',''))[:80]}",
            "tool_calls":[], "stop_reason":"end"}


def _to_anthropic(messages):
    out=[]
    for m in messages:
        if m["role"]=="user":
            out.append({"role":"user","content":m["content"]})
        elif m["role"]=="assistant":
            blocks=[]
            if m.get("content"): blocks.append({"type":"text","text":m["content"]})
            for tc in m.get("tool_calls",[]):
                blocks.append({"type":"tool_use","id":tc["id"],"name":tc["name"],"input":tc["args"]})
            out.append({"role":"assistant","content":blocks})
        elif m["role"]=="tool":
            out.append({"role":"user","content":[{"type":"tool_result",
                        "tool_use_id":m["tool_call_id"],"content":str(m["content"])}]})
    return out

def _to_openai(messages, system):
    out=[]
    if system: out.append({"role":"system","content":system})
    for m in messages:
        if m["role"]=="user":
            out.append({"role":"user","content":m["content"]})
        elif m["role"]=="assistant":
            msg={"role":"assistant","content":m.get("content") or None}
            if m.get("tool_calls"):
                msg["tool_calls"]=[{"id":tc["id"],"type":"function","function":{
                    "name":tc["name"],"arguments":json.dumps(tc["args"])}} for tc in m["tool_calls"]]
            out.append(msg)
        elif m["role"]=="tool":
            out.append({"role":"tool","tool_call_id":m["tool_call_id"],"content":str(m["content"])})
    return out

def call_llm(messages, tools=None, system=None, max_tokens=1024, temperature=0):
    '''One function. Any provider. This is portable across whatever API key you have or your institution has.'''
    if PROVIDER=="mock":
        return _mock_call(messages, tools)
    if PROVIDER=="anthropic":
        from anthropic import Anthropic
        client=Anthropic()
        kw=dict(model=MODELS["anthropic"], max_tokens=max_tokens,
                temperature=temperature, messages=_to_anthropic(messages))
        if system: kw["system"]=system
        if tools: kw["tools"]=[{"name":t["name"],"description":t["description"],
                                "input_schema":t["parameters"]} for t in tools]
        r=client.messages.create(**kw)
        text=""; calls=[]
        for b in r.content:
            if b.type=="text": text+=b.text
            elif b.type=="tool_use": calls.append({"id":b.id,"name":b.name,"args":b.input})
        return {"text":text,"tool_calls":calls,"stop_reason":r.stop_reason}
    if PROVIDER=="openai":
        from openai import OpenAI
        client=OpenAI()
        kw=dict(model=MODELS["openai"], messages=_to_openai(messages, system),
                temperature=temperature, max_completion_tokens=max_tokens)
        if tools: kw["tools"]=[{"type":"function","function":{"name":t["name"],
                    "description":t["description"],"parameters":t["parameters"]}} for t in tools]
        r=client.chat.completions.create(**kw)
        msg=r.choices[0].message
        calls=[{"id":tc.id,"name":tc.function.name,
                "args":json.loads(tc.function.arguments or "{}")} for tc in (msg.tool_calls or [])]
        return {"text":msg.content or "", "tool_calls":calls,
                "stop_reason":"tool_use" if calls else "end"}
    raise ValueError("Unknown PROVIDER: "+PROVIDER)

print(f"Setup ready. PROVIDER={PROVIDER!r}.")

In [ ]:
def run_agent(user_task, tools_spec, tool_fns, system=None, max_steps=6, verbose=True):
    '''The agent loop. The idea is:
       plan -> act (maybe call a tool) -> observe (feed the result back) -> repeat,
       until the model returns a final answer instead of a tool call.'''
    messages = [{"role":"user","content":user_task}]
    for step in range(1, max_steps+1):
        reply = call_llm(messages, tools=tools_spec, system=system)
        if reply["tool_calls"]:                                  # the model wants a tool
            messages.append({"role":"assistant","content":reply["text"],
                             "tool_calls":reply["tool_calls"]})
            for tc in reply["tool_calls"]:
                if verbose: print(f"  step {step}: tool `{tc['name']}` <- {tc['args']}")
                try:    result = tool_fns[tc["name"]](**tc["args"])   # run the real function
                except Exception as e: result = f"ERROR: {e}"
                messages.append({"role":"tool","tool_call_id":tc["id"],
                                 "name":tc["name"],"content":result})   # observe
        else:                                                    # no tool -> we are done
            if verbose: print(f"  step {step}: final answer")
            return reply["text"], messages
    return "Stopped: hit max_steps.", messages

# What is a tool?

Two parts, no technical jargon:
1. A **python function** that does the real, actual work.
2. **A Schema**(name, description and arguments) that tells the model the tool exists. The model reass the description to decide *when* to call it.

# Step 2 - Tool 1: Web search

Here we are exploring 2 ways:
- Default is DDGS (the `ddgs` library): It requires no signup, api required.
- **Tavily** API key 

In [ ]:
def search_web(query, k=3):
    # search the live web and gives short snippets. If tavily key available
    # it will be used, else DuckDuckGo (This is keyless)

    if os.environ.get("TAVILY_API_KEY"):
        try:
            from tavily import TavilyClient
            res = TavilyClient().search(query, max_results=k)
            return "\n".join(f"- {r['title']}: {r['content'][:250]}" for r in res["results"])
        except Exception as e:
            return f"(Tavily error: {e})"
        
    try:
        from ddgs import DDGS
        with DDGS() as d:
            hits = list(d.text(query, max_results=k))
        if hits:
            return "\n".join(f"- {h['title']}: {h['body'][:250]}" for h in hits)
    except Exception as e:
        return f"(web search unavailable offline: {e})"
    return "(no results)"

# Quick smoke test (works only with a network; fine if it prints the offline note)
print(search_web("OWASP guidelines"))

## Step 4 - Tool 2: document retrieval

This searches the local files. For now it's a transparent **keyword score**: count how many query words appear in each document, return the best matches. In Notebook 3, when we upgrade this to real semantic search with embeddings, so you see the difference.

For now, the repo holds a folder `../datasets` with synthetic research-support documents

In [ ]:
import glob, os, re

def load_corpus(folder="../datasets"):
    docs = {}
    for path in glob.glob(os.path.join(folder, "*.md")) + glob.glob(os.path.join(folder, "*.txt")):
        name = os.path.basename(path)
        if name.lower() == "readme.md":
            continue
        with open(path) as f:
            docs[name] = f.read()
    if not docs:
        docs = {"inline_note.md": "Synthetic note: log every query for a reproducible literature triage; measure retrieval precision and recall."}
    return docs

CORPUS = load_corpus()
print("Loaded", len(CORPUS), "documents:", list(CORPUS))

def retrieve_docs(query, k=2):
    #Return the k local documents whose text best matches the query and keyword score
    q = set(re.findall(r"[a-z]+", query.lower()))
    scored = []
    for name, text in CORPUS.items():
        words = re.findall(r"[a-z]+", text.lower())
        overlap = sum(1 for w in words if w in q)
        scored.append((overlap, name, text))
    scored.sort(reverse=True)
    top = [s for s in scored if s[0] > 0][:k]
    if not top:
        return "(no local document matched)"
    return "\n\n".join(f"[{name}] (score {score})\n{text[:280]}..." for score, name, text in top)

print("\nSample retrieval for 'AI funding deadline':\n")
print(retrieve_docs("AI funding deadline proposal"))

## Step 4 - Tool 3: structured outputs

Free text is hard for code to consume. Often you want the agent to return **JSON with a fixed shape** that you can drop straight into a spreadsheet or database. We will define the shape as a schema, then validate what comes back. If it doesn't fit the shape, we know immediately instead of discovering it later.

In [ ]:
def validate_record(obj, required=("title","deadline","award_max")):
    #structural check. On a larger scale, use jsonschema or pydantic.
    missing = [f for f in required if f not in obj]
    return (len(missing) == 0, missing)

# structured pull fields froma retrieved document
extract_schema = {
    "type":"object",
    "properties":{
        "title":{"type":"string"},
        "deadline":{"type":"string","description":"ISO date if present"},
        "award_max":{"type":"string"},
        },
    "required":["title","deadline","award_max"],
}
print("Schema the model must fill:\n", json.dumps(extract_schema, indent=2))

## Step 5 - the agent choice

Now give all three tools and give the agent a task that needs more than one tool. Then watch how the model reads the descriptions and picks a tool per step.

In [ ]:
tools_spec = [
    {"name":"search_web",
     "description":"Search the live web for current information and return snippets. Use for anything time-sensitive or not in local documents.",
     "parameters":{"type":"object","properties":{
        "query":{"type":"string"}, "k":{"type":"integer","description":"how many results"}},
        "required":["query"]}},
    {"name":"retrieve_docs",
     "description":"Search the local (institutional or system) documents (funding notes, policies, method notes). Use for internal/administrative questions.",
     "parameters":{"type":"object","properties":{
        "query":{"type":"string"}, "k":{"type":"integer"}}, "required":["query"]}},
]
tool_fns = {"search_web": search_web, "retrieve_docs": retrieve_docs}

mock_reset([
    ("tool", {"name":"retrieve_docs", "args":{"query":"NSF AI funding deadline award", "k":1}}),
    ("final", "From the local NSF note: full proposals are due November 3, 2026; "
              "awards up to $600,000 over 3 years. (Retrieved from the datasets folder.)"),
])

print("AGENT deciding which tool to use:")
answer, transcript = run_agent(
    "What's the deadline and award size for the NSF AI infrastructure program? Check our local documents.",
    tools_spec, tool_fns)
print("\nAnswer:\n", answer)

# Exercise - ADD your own Tool

Steps:
1. Write the Python function.
2. Write a clear schema (remember: the description is what the model reads to choose it).
3. Add it to `tools_spec` and `tool_fns`.
4. Ask a question that needs it. In mock mode, script the call; with a real key, let the model choose.

## Reflection
- Which of your daily tasks are really "the model needs to *look something up* or *do* something"? Those can be tools.
- A tool's description is a tiny piece of writing that changes how the agent reacts. Where else do you need precise wording in your work?